# 17 — Fair Value Gap (FVG)

## Goal
Identify **Fair Value Gaps** — 3-candle patterns where candle-1 and candle-3 leave a price gap on candle-2.  
FVGs mark areas where price moved so fast that no trading occurred, and they act as **refined entry points** inside a wider zone.

---

## Definition

### Bullish FVG (demand imbalance)

$$\text{low}_{i+1} > \text{high}_{i-1}$$

Candle $i-1$'s high and candle $i+1$'s low do not overlap — a gap above candle $i-1$.

### Bearish FVG (supply imbalance)

$$\text{high}_{i+1} < \text{low}_{i-1}$$

Candle $i-1$'s low and candle $i+1$'s high do not overlap — a gap below candle $i-1$.

---

## Gap box

| FVG type | Box top | Box bottom |
|----------|---------|------------|
| Bullish  | $\text{low}_{i+1}$ | $\text{high}_{i-1}$ |
| Bearish  | $\text{low}_{i-1}$ | $\text{high}_{i+1}$ |

Price tends to return to fill this gap before continuing.  
**Unfilled FVGs** inside a zone provide the sharpest entry.

---

## Visualization
FVG boxes drawn as thin rectangles on the price chart — blue for bullish, orange for bearish.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. `find_fvgs` — scan for all fair value gaps

In [ ]:
def find_fvgs():
    bullish, bearish = [], []
    for i in range(1, len(df) - 1):
        # bullish: low[i+1] > high[i-1]
        if l_arr[i+1] > h_arr[i-1]:
            bullish.append({"bar": i, "top": l_arr[i+1], "bottom": h_arr[i-1]})
        # bearish: high[i+1] < low[i-1]
        if h_arr[i+1] < l_arr[i-1]:
            bearish.append({"bar": i, "top": l_arr[i-1], "bottom": h_arr[i+1]})
    return bullish, bearish

bull_fvg, bear_fvg = find_fvgs()
print(f"Bullish FVGs: {len(bull_fvg)}   Bearish FVGs: {len(bear_fvg)}")


## 3. Visualization — FVG boxes on price chart

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
))

for fvg in bull_fvg:
    x0 = df.index[fvg["bar"] - 1]
    x1 = df.index[min(fvg["bar"] + 1, len(df) - 1)]
    fig.add_vrect(x0=x0, x1=x1, y0=fvg["bottom"], y1=fvg["top"],
                  fillcolor="rgba(124,131,253,0.18)",
                  line=dict(color="#7c83fd", width=0.8))

for fvg in bear_fvg:
    x0 = df.index[fvg["bar"] - 1]
    x1 = df.index[min(fvg["bar"] + 1, len(df) - 1)]
    fig.add_vrect(x0=x0, x1=x1, y0=fvg["bottom"], y1=fvg["top"],
                  fillcolor="rgba(255,152,0,0.15)",
                  line=dict(color="#ff9800", width=0.8))

# legend stubs
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(size=10, color="#7c83fd", symbol="square"),
    name=f"Bullish FVG ({len(bull_fvg)})"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(size=10, color="#ff9800", symbol="square"),
    name=f"Bearish FVG ({len(bear_fvg)})"))

fig.update_layout(
    title="Fair Value Gaps (FVG)",
    height=540, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
